# 05m. GLiClass 퓨샷 파인튜닝: 모델 선택판 (훈련 / 검증 / 시험)

`finetuning.ipynb`와 같은 코드에 **모델(`MODEL`)을 하이퍼파라미터로** 두고, 결과를 **모델 이름 폴더**에 저장한다. 그 밖의 설정과 지표는 `finetuning.ipynb`와 같다.

**목적**: 04에서 나눈 데이터로 GLiClass edge 모델을 퓨샷 파인튜닝하고, 검증 세트로 에폭을 고른 뒤 시험 세트에서 한 번만 평가한다. reference의 시범 코드(`reference/pilot1.ipynb`, `pilot2.ipynb`)를 바탕으로, 검증하면서 찾은 문제를 고쳤다.

**A / B**: 압축 방식이 다른 두 데이터. A = 윈도우로 나눈 뒤 같은 내용 윈도우 병합, B = 윈도우 전에 이벤트 반복 압축 (`preprocess.ipynb`, `window.ipynb`)

**입력**: `04_split_v2.ipynb`가 만든 `data/v2/{A,B}_{tactic,technique,both}_k{k}/` (`VARIANT = 'B_both_k16'`처럼 폴더 이름)
- 전술·기법은 MITRE ID 하나가 라벨 하나다 (labels/README.md L1). 묶은 라벨(`TA0003+TA0004`)이 있는 예전 분할은 3번 칸에서 멈춘다
- `labels.csv`: 라벨 문장, 종류(`kind`), 학습/숨김(`role`)과 숨긴 이유(`role_reason`), pool/val/test 개수
- `splits/{VARIANT}/kshot/train_k{k}_s{seed}.json`: 학습 파일 (학습 라벨마다 k개 이상, 후보 라벨 = 학습 라벨)
- `splits/{VARIANT}/val_gliclass.json`, `val.jsonl`: 검증 (후보 = 학습 라벨). 에폭·임계값 고르기에만 쓴다
- `splits/{VARIANT}/test_gliclass.json`, `test.jsonl`: 시험 (후보 = 학습 + 숨김 라벨). 마지막에 한 번만 쓴다

**숨김 라벨 두 종류** (`role_reason`으로 나눔)
- `held`: k와 상관없이 늘 숨김. 일부러 숨긴 전술(TA0040, TA0112)과 pool/val/test 중 하나에 없어서 숨긴 라벨. **k끼리 제로샷 성능을 비교할 때 이것을 본다**
- `heldk`: 이 k에서만 숨김 (`pool<k`: pool에 k개가 없음). k가 커질수록 늘어난다

**출력** (`data/v2/{VARIANT}/results/{모델 이름}/finetune_{VARIANT}{TAG}/`, 모델 이름 = `MODEL`의 `/` 뒤. 예: `gliclass-modern-large-v3.0`)
- `runs.csv`: 실행(k, seed)마다 고른 에폭과 시험 점수 (k = 0은 파인튜닝 전 기준선)
- `curves.csv`: 에폭마다 학습 손실, 검증 손실, 검증 지표
- `per_label.csv`: 라벨별 시험 점수
- 그래프 3개 (`fig_*.png`)
- `KEEP_MODELS = True`면 고른 에폭의 모델: `{WORK}/best/{모델 이름}/{VARIANT}{TAG}_k{k}_s{seed}/`

**reference 시범 코드에서 고친 점**

| # | 시범 코드 | 문제 | 이 노트북 |
|---|---|---|---|
| 1 | `--val_data_path`에 학습 파일을 넣고 `save_steps`를 크게 해서 검증을 끔 | 검증 세트가 없어 에폭·학습률을 고를 근거가 없음. 시험 점수로 고르면 시험 누수 | 04의 `val_gliclass.json`으로 에폭마다 검증, 가장 좋은 에폭을 고른 뒤 시험은 한 번만 |
| 2 | seed는 예시 뽑기에만 씀 | `train.py`에 시드 인자가 없고, Trainer가 시드를 늘 42로 다시 고정함 | `train.py`를 감싸서 Trainer가 쓰는 시드(`TrainingArguments.seed`)를 우리 seed로 바꿈 |
| 3 | 1등 라벨 하나만 정답과 비교 (단일 라벨) | 우리 윈도우는 라벨이 여러 개 붙음 (전술 여러 개 + 기법 + 상위 기법) | 라벨마다 임계값으로 판정하는 멀티라벨 지표 (macro/micro F1, 라벨별 F1, PR-AUC) + 1등 정확도도 같이 |
| 4 | 손실을 출력 문자열에서 정규식으로 뽑음 | 형식이 바뀌면 조용히 NaN | 체크포인트의 `trainer_state.json` 기록에서 읽음 |
| 5 | `train.py`가 출력하는 accuracy/F1을 볼 수 있음 | 라벨 칸을 전부 펼쳐 weighted 평균을 내서, 대부분 0인 칸 덕분에 항상 높게 나옴 | 무시하고 우리 지표로 평가 (검증 손실만 사용) |

**준비**: Colab은 GPU 런타임. 드라이브에 `data/v2/`(또는 `ctgli_splits*.zip`)를 올리거나, 환경 변수 `DATA_DIR`로 데이터 폴더를 지정한다. 로컬은 저장소의 `data/`를 자동으로 쓴다. 처음에는 `SMOKE = True`로 몇 분 안에 끝까지 도는지 먼저 확인한다.

**칸마다 보는 법**: `[하는 일]` / `[볼 것]` / `[확인 질문]` (02~04와 같음)

## 1. 설치

[하는 일] GLiClass를 시범 코드와 같은 커밋으로 설치하고, 학습 스크립트 `train.py`가 있는 저장소를 받는다. 이미 있으면 건너뛴다.

설치 뒤 다음 칸에서 import 에러가 나면: 런타임 > 세션 다시 시작 후 **2번부터** 다시 실행.

In [ ]:
import os, sys, subprocess, importlib.util

COMMIT = '6a00f8067648f25aa0f4f477d167c5f391cb4165'   # reference 시범 코드와 같은 버전

try:                                                  # find_spec('google.colab')은 google 패키지가 없으면 오류
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if 'DATA_DIR' in os.environ:                          # 경로를 직접 지정 (여러 설정을 차례로 돌릴 때): 드라이브 연결 건너뜀
    ROOT = os.environ['DATA_DIR']
    WORK = os.environ.get('WORK_DIR', '/content/work' if IN_COLAB else os.path.join(ROOT, 'finetune_work'))
else:
  try:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT, WORK = '/content/drive/MyDrive', '/content/work'
    # 드라이브에 ctgli_splits.zip만 올렸으면 그 자리에 푼다 (splits/와 labels.csv가 생김)
    import glob, zipfile
    if not glob.glob(f'{ROOT}/**/splits/*/val_gliclass.json', recursive=True):
        for z in glob.glob(f'{ROOT}/**/ctgli_splits*.zip', recursive=True):
            zipfile.ZipFile(z).extractall(os.path.dirname(z)); print('압축 풀기:', z)
  except ImportError:                                 # 로컬: 저장소의 data/ (04_split_v2 결과)
    from pathlib import Path
    REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
    ROOT = os.environ.get('DATA_DIR', str(REPO_ROOT / 'data'))
    WORK = os.environ.get('WORK_DIR', os.path.join(ROOT, 'finetune_work'))
os.makedirs(WORK, exist_ok=True)

need = [p for p, m in [(f'git+https://github.com/Knowledgator/GLiClass.git@{COMMIT}', 'gliclass'),
                       ('datasets', 'datasets'), ('accelerate', 'accelerate')] if importlib.util.find_spec(m) is None]
if need:                                              # accelerate: train.py의 Trainer가 필요로 함
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *need], check=True)
REPO = os.path.join(WORK, 'gliclass_repo')
if not os.path.exists(f'{REPO}/train.py'):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/Knowledgator/GLiClass.git', REPO], check=True)
    subprocess.run(['git', '-C', REPO, 'checkout', '-q', COMMIT], check=True)
print('train.py:', os.path.exists(f'{REPO}/train.py'), '| WORK =', WORK)

## 2. 설정

[하는 일] 실험 조건을 한 곳에 모은다. 바꿔볼 값은 여기만 고치면 된다.

- `MODEL`: GLiClass 모델 (Hugging Face 이름). 예: `knowledgator/gliclass-edge-v3.0`, `knowledgator/gliclass-modern-large-v3.0`. 4번 칸이 이 모델의 토크나이저로 입력 길이를 검사한다
- `VARIANT`: 분할 폴더 이름 `{A|B}_{tactic|technique|both}_k{k}` (예: `B_both_k16`)
- `K_LIST`: `None`이면 폴더에 있는 k-shot 파일의 k를 모두 씀 (v2 폴더는 k가 하나)
- `EPOCHS`: 최대 에폭. 에폭마다 체크포인트를 남기고 검증으로 가장 좋은 에폭을 고른다
- `SELECT_BY`: 에폭을 고르는 검증 지표
- `VAL_BENIGN_N`: 검증에서 쓸 **정상만 있는 윈도우** 수. 공격 윈도우는 개수와 상관없이 전부 쓴다 (무작위로 줄이면 드문 학습 라벨이 검증에서 빠져 에폭·임계값을 못 고름)
- `KEEP_MODELS`: 고른 에폭의 모델을 남길지 (실행 하나에 약 150MB)
- `SMOKE = True`: k 하나, seed 0, 2에폭, 평가 일부만. 끝까지 도는지 확인용 (몇 분)

In [ ]:
import json, math, glob, shutil, time, random
import numpy as np, pandas as pd, torch

VARIANT = 'B_tactic_k16'     # 분할 폴더 이름 (data/v2/ 아래)
MODEL = 'knowledgator/gliclass-edge-v3.0'   # 하이퍼파라미터: GLiClass 모델 (Hugging Face 이름)
K_LIST = None              # None이면 폴더에 있는 k 전부
SEEDS = [0, 1, 2]
EPOCHS = 10
ENCODER_LR = 2e-5          # train.py 기본값
OTHERS_LR = 2e-5           # train.py 기본값은 5e-5. 시범 코드는 인코더와 같게 둠
BATCH = 8
MAX_LEN = 1024             # 4번에서 실제 토큰 수로 확인
FOCAL_ALPHA, FOCAL_GAMMA = 0.5, 2.0
THRESHOLD = 0.5            # 고정 임계값 (비교용 *_f1_05 지표)
TUNE_THRESHOLD = True      # 라벨마다 검증 세트에서 F1이 가장 좋은 임계값을 골라 시험에 적용
SELECT_BY = 'val_macro_ap' # 에폭 고르는 기준. 임계값과 무관한 PR-AUC (F1@0.5로 고르면 점수가 0.5 아래로 내려간 에폭을 못 고름)
TRAINER_EVAL_N = 500       # train.py가 에폭마다 검증 손실을 낼 때 쓰는 검증 개수
AGNEWS_N = 500             # 원래 실력 확인용 ag_news 개수
KEEP_MODELS = True         # 고른 에폭의 모델을 {WORK}/best/에 남김 (실행 하나에 약 150MB)
EXTRA_ARGS = []            # train.py에 더 넘길 인자. 예: ['--use_ewc', '--prev_data_path', 'prev.json']
TAG = ''                   # 결과 폴더 이름 뒤에 붙음. 설정을 바꿔 비교할 때 (예: '_lr1e-4')

SMOKE = False
VAL_BENIGN_N = 300         # 검증의 정상만 있는 윈도우 수 (공격 윈도우는 전부). None이면 검증 전부
TEST_N = None              # 시험은 전부 (None)
if SMOKE:
    SEEDS, EPOCHS = [0], 2           # k는 폴더에 있는 것 중 첫 번째만 (3번 칸)
    VAL_BENIGN_N, TEST_N, AGNEWS_N, TRAINER_EVAL_N = 50, 300, 100, 100

MODEL_NAME = MODEL.rstrip('/').split('/')[-1]   # 결과·체크포인트 폴더 이름
DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print('모델:', MODEL, '| 결과 폴더 이름:', MODEL_NAME)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '없음 (CPU로는 매우 느림)')

## 3. 데이터 불러오기 (04 결과)

[하는 일] `splits/{VARIANT}`를 찾고 (결과는 `results/{MODEL_NAME}/` 아래), 그 두 단계 위의 `labels.csv`를 읽는다. `labels.csv`의 `kind` 열로 라벨 방식을 정한다.

| `kind`에 있는 것 | 방식 | 평가 |
|---|---|---|
| tactic (+ benign) | 전술 | 한 번 |
| technique (+ benign) | 기법 | 한 번 |
| tactic, technique (+ benign) | 전술+기법 | **학습은 함께, 추론·채점은 전술과 기법을 나눠서** (지표 이름 앞에 `tactic_`, `technique_`) |

`kind` 열이 없거나 묶은 라벨(`TA0003+TA0004`)이 있으면 멈춘다 (예전 라벨 체계는 쓰지 않음).

[볼 것] pool/val/test 개수가 04의 결과와 같은지.

In [ ]:
split_hits = glob.glob(f'{ROOT}/**/splits/{VARIANT}/val_gliclass.json', recursive=True)
assert split_hits, f'splits/{VARIANT}를 못 찾음 (04 먼저)'
assert len(split_hits) == 1, f'같은 이름의 분할이 여러 개: {split_hits}'
SPLIT = os.path.dirname(split_hits[0])
DATA_DIR = os.path.dirname(os.path.dirname(SPLIT))            # labels.csv는 splits/ 바로 위
assert os.path.exists(f'{DATA_DIR}/labels.csv'), f'{DATA_DIR}/labels.csv 없음'
OUT = f'{DATA_DIR}/results/{MODEL_NAME}/finetune_{VARIANT}{TAG}' + ('_smoke' if SMOKE else '')   # 모델 이름 폴더
os.makedirs(OUT, exist_ok=True)

labels = pd.read_csv(f'{DATA_DIR}/labels.csv')
# 라벨 규칙 (labels/README.md L1): 전술·기법은 MITRE ID 하나가 라벨 하나. 묶은 라벨(TA0003+TA0004)은 쓰지 않는다
assert 'kind' in labels, f'{DATA_DIR}/labels.csv에 kind 열이 없음: 예전 라벨 표. 04_split_v2.ipynb로 만든 분할을 쓸 것'
assert not labels.label_id.str.contains(r'\+').any(), f'묶은 라벨이 있음: {labels.label_id[labels.label_id.str.contains(r"[+]")].tolist()}'
TRAIN = labels[labels.role == 'train'].label_id.tolist()
HELD = labels[labels.role == 'held_out'].label_id.tolist()
# 숨김 두 종류: held = k와 상관없이 늘 숨김 (held_tactic, no_pool/no_val/no_test), heldk = 이 k에서만 (pool<k)
assert 'role_reason' in labels, 'labels.csv에 role_reason 열이 없음 (04_split_v2 다시)'
HELD_K = labels[(labels.role == 'held_out') & labels.role_reason.astype(str).str.contains('pool<')].label_id.tolist()
HELD_FIX = [l for l in HELD if l not in HELD_K]
name = dict(zip(labels.label_id, labels.name))
text_of = dict(zip(labels.label_id, labels.label_text))
id_of = {t: l for l, t in text_of.items()}
KIND_OF = dict(zip(labels.label_id, labels.kind))
KINDS = [k for k in ['tactic', 'technique'] if k in set(labels.kind)]
PREFIX = {k: '' if len(KINDS) == 1 else f'{k}_' for k in KINDS}   # 전술+기법이면 지표 이름 앞에 종류


def of_kind(lids, kind):
    """lids 중 이 종류(+ BENIGN)인 라벨."""
    return [l for l in lids if KIND_OF[l] in (kind, 'benign')]


parts = {sp: pd.read_json(f'{SPLIT}/{sp}.jsonl', lines=True) for sp in ['pool', 'val', 'test']}
gl = {sp: json.load(open(f'{SPLIT}/{sp}_gliclass.json')) for sp in ['val', 'test']}
train_files = {}
for f in sorted(glob.glob(f'{SPLIT}/kshot/train_k*_s*.json')):
    k, s = map(int, os.path.basename(f)[7:-5].split('_s'))
    train_files[(k, s)] = f
if K_LIST is None:
    K_LIST = sorted({k for k, _ in train_files})
if SMOKE:
    K_LIST = K_LIST[:1]

print('SPLIT =', SPLIT, '| 결과 =', OUT)
print({sp: len(d) for sp, d in parts.items()}, '| 방식:', '+'.join(KINDS))
for kind in KINDS:
    print(f'[{kind}] 학습 {of_kind(TRAIN, kind)}')
    print(f'[{kind}] 숨김 held (늘)  {[l for l in HELD_FIX if KIND_OF[l] == kind]}')
    print(f'[{kind}] 숨김 heldk (이 k) {[l for l in HELD_K if KIND_OF[l] == kind]}')
print('k-shot 파일:', sorted(train_files), '| 사용할 k:', K_LIST)

## 4. 데이터 검사

[하는 일] 학습 전에 데이터가 규칙대로인지 확인한다. 하나라도 틀리면 멈춘다.

1. 평가 파일(gliclass 형식)과 jsonl의 개수·문장이 같은지
2. 학습·검증 파일의 후보 라벨이 학습 라벨뿐인지 (숨긴 라벨 문장이 섞이면 제로샷 실험이 깨짐)
3. 시험 파일의 후보 라벨이 학습 + 숨김 라벨인지
4. 학습 문장이 검증·시험에 없는지 (누수)
5. 파일 안의 라벨 문장이 지금 `labels.csv`와 같은지 (labels.csv를 고친 뒤 04를 안 돌린 경우)
6. k-shot 파일마다 라벨별 개수가 k 이상인지 (모자라면 경고. B_cap처럼 상한을 둔 파일)
7. 토큰 길이가 `MAX_LEN` 안에 드는지: 학습(학습 라벨 전부)과 추론(종류별 학습 + 숨김 라벨) 각각

[볼 것] 마지막 줄에 `검사 통과`.

In [ ]:
from transformers import AutoTokenizer

for sp in ['val', 'test']:
    assert len(gl[sp]) == len(parts[sp]), f'{sp}: gliclass {len(gl[sp])} vs jsonl {len(parts[sp])}'
    assert [r['text'] for r in gl[sp]] == parts[sp].text.tolist(), f'{sp}: 문장 순서 다름'

train_texts = {text_of[l] for l in TRAIN}
assert all(set(r['all_labels']) == train_texts for r in gl['val']), 'val 후보 라벨이 학습 라벨이 아님'
assert all(set(r['all_labels']) == {text_of[l] for l in TRAIN + HELD} for r in gl['test']), 'test 후보 라벨이 학습 + 숨김 라벨이 아님'

seen_eval = set(parts['val'].text) | set(parts['test'].text)
rows = []
for (k, s), f in sorted(train_files.items()):
    tr = json.load(open(f))
    assert all(set(r['all_labels']) == train_texts for r in tr), f'{f}: 후보 라벨에 숨긴 라벨'
    assert not ({r['text'] for r in tr} & seen_eval), f'{f}: 학습 문장이 검증/시험에 있음'
    assert all(t in id_of for r in tr for t in r['true_labels']), f'{f}: labels.csv에 없는 라벨 문장 (04 다시)'
    cnt = pd.Series([id_of[t] for r in tr for t in r['true_labels']]).value_counts()
    short = {name[l]: int(cnt.get(l, 0)) for l in TRAIN if cnt.get(l, 0) < k}
    if short:                                          # 상한 적용 파일 (예: B_cap): 모자란 라벨은 pool에 있는 만큼만
        print(f'주의 k={k} seed={s}: {k}개 미만인 라벨 {short}')
    rows.append({'k': k, 'seed': s, 'windows': len(tr), **{name[l]: int(cnt.get(l, 0)) for l in TRAIN}})
print(pd.DataFrame(rows).to_string(index=False))

tok = AutoTokenizer.from_pretrained(MODEL)
def lab_tok(lids):
    return sum(len(tok(text_of[l], add_special_tokens=False)['input_ids']) + 1 for l in lids)

txt_tok = pd.concat([parts[sp].text for sp in parts]).map(lambda t: len(tok(t)['input_ids']))
need = {'학습': lab_tok(TRAIN), **{f'추론 {k}': lab_tok(of_kind(TRAIN + HELD, k)) for k in KINDS}}
print(f'\n토큰: 문장 평균 {txt_tok.mean():.0f} / 최대 {txt_tok.max()}')
for what, n in need.items():
    print(f'  {what}: 라벨 {n} + 문장 최대 {txt_tok.max()} = {txt_tok.max() + n} (MAX_LEN {MAX_LEN})')
    assert txt_tok.max() + n <= MAX_LEN, f'{what}: MAX_LEN을 늘리기 (넘치면 문장 뒤쪽이 잘림)'
print('검사 통과')

## 5. 평가 함수

[하는 일] 모델 하나를 받아 라벨마다 sigmoid 점수를 내고, 여러 지표를 계산한다.

**종류별로 따로 추론한다.** 전술+기법 방식이면 전술 후보만 넣어 한 번, 기법 후보만 넣어 한 번 추론하고 따로 채점한다 (지표 이름 앞에 `tactic_`, `technique_`). 학습은 둘을 함께 한다. 전술만 / 기법만 학습한 모델과 같은 조건으로 비교할 수 있고, 후보 라벨이 섞여 서로 점수를 다투지 않는다.

| 이름 | 데이터 / 후보 라벨 | 뜻 |
|---|---|---|
| `val_*` | 검증 / 학습 라벨 | 에폭 고르기용 |
| `seen_*` | 시험 중 숨김 라벨이 없는 윈도우 / 학습 라벨 | 학습한 라벨 성능 |
| `held_*` | 시험 중 **늘 숨긴** 라벨이 있는 윈도우 / 그 라벨 + BENIGN | 처음 보는 라벨 (제로샷). k와 상관없이 같은 라벨이라 k끼리 비교 가능 |
| `heldk_*` | 시험 중 **이 k에서만 숨긴** 라벨(pool<k)이 있는 윈도우 / 그 라벨 + BENIGN | k마다 라벨이 달라 k끼리 비교하지 않음 |
| `all_*` | 시험 전체 / 학습 + 숨김 | 실제 상황처럼 전부 놓고 |
| `agnews_acc` | ag_news 일반 뉴스 4분류 | 원래 실력이 남았나 |

지표 (정답이 있는 라벨만 평균): `macro_f1`(라벨별 F1 평균, 주 지표), `micro_f1`, `macro_ap`(임계값 없는 PR-AUC), `top1_acc`(1등 라벨이 정답 중 하나인 비율), `attack_recall`/`fpr`/`separation`(= recall - FPR)

**공격 판정**: 공격 라벨 중 **하나라도 임계값을 넘으면 공격**. 정답은 라벨에 공격(BENIGN이 아닌 것)이 하나라도 있으면 공격. 같은 내용 윈도우를 합치면서 `BENIGN`과 공격이 함께 붙을 수 있어서 (README L6), 1등 라벨이 BENIGN인지로는 판정하지 않는다.

**임계값**: 파인튜닝하면 점수 전체가 0.5 아래로 내려가는 경우가 많아서, 0.5 고정이면 순위는 맞는데 F1이 0이 된다. 그래서
- 검증은 공격 윈도우를 전부 쓰고 정상만 있는 윈도우를 `VAL_BENIGN_N`개 뽑는다 (학습 라벨이 검증에서 빠지지 않게)
- `*_macro_f1`: 라벨마다 **검증 세트**에서 F1이 가장 좋은 임계값을 골라 적용 (시험 데이터는 임계값 고르기에 안 씀). 종류별로 따로
- 숨김 라벨은 검증 데이터가 없으므로 같은 종류 학습 공격 라벨 임계값의 평균을 씀
- `*_macro_f1_05`: 0.5 고정 (비교용)

In [ ]:
from sklearn.metrics import precision_recall_fscore_support, average_precision_score
from datasets import load_dataset
from gliclass import GLiClassModel, ZeroShotClassificationPipeline

agnews = load_dataset('fancyzhx/ag_news', split='test').shuffle(seed=0).select(range(AGNEWS_N))
AG_LABELS = ['World', 'Sports', 'Business', 'Sci/Tech']

def subset(d, n, seed=0):
    return d if n is None or len(d) <= n else d.sample(n, random_state=seed)

def val_subset(d, n_benign, seed=0):
    """공격 윈도우는 전부, 정상만 있는 윈도우는 n_benign개."""
    benign_only = d['labels'].apply(lambda ls: all(l == 'BENIGN' for l in ls))
    return pd.concat([d[~benign_only], subset(d[benign_only], n_benign, seed)]).sort_index()

val_df = val_subset(parts['val'], VAL_BENIGN_N).reset_index(drop=True)
test_df = subset(parts['test'], TEST_N).reset_index(drop=True)
GROUPS = {}                                    # 종류마다: 학습·숨김 라벨, 시험 seen/held/heldk 윈도우
for kind in KINDS:
    tr = of_kind(TRAIN, kind)
    hf = [l for l in HELD_FIX if KIND_OF[l] == kind]
    hk = [l for l in HELD_K if KIND_OF[l] == kind]
    has = lambda lids: test_df['labels'].apply(lambda ls: any(l in lids for l in ls))
    GROUPS[kind] = dict(train=tr, held=hf, heldk=hk,
                        seen_df=test_df[~has(hf + hk)].reset_index(drop=True),   # 숨김 라벨이 하나도 없는 윈도우
                        held_df=test_df[has(hf)].reset_index(drop=True),
                        heldk_df=test_df[has(hk)].reset_index(drop=True))

def load_pipe(path):
    model = GLiClassModel.from_pretrained(path)
    tk = AutoTokenizer.from_pretrained(path)
    return ZeroShotClassificationPipeline(model, tk, classification_type='multi-label', device=DEVICE,
                                          max_length=MAX_LEN, progress_bar=False)

def score_matrix(pipe, texts, lids):
    out = pipe(list(texts), [text_of[l] for l in lids], threshold=0.0, batch_size=16)
    return np.array([[{x['label']: x['score'] for x in r}[text_of[l]] for l in lids] for r in out])

def label_matrix(df, lids):
    return np.array([[l in ls for l in lids] for ls in df['labels']], dtype=int)

def tune_thresholds(df, S, lids):
    """라벨마다 F1이 가장 좋은 임계값 (검증 세트에서만 부름)."""
    Y, th = label_matrix(df, lids), {}
    for j, l in enumerate(lids):
        if Y[:, j].sum() == 0:
            continue
        cand = np.unique(np.quantile(S[:, j], np.linspace(0, 1, 201)))
        f = [precision_recall_fscore_support(Y[:, j], S[:, j] >= t, average='binary', zero_division=0)[2] for t in cand]
        th[l] = float(cand[int(np.argmax(f))])
    return th

def thresholds_for(lids, th):
    attack = [th[l] for l in th if l != 'BENIGN']
    unseen = float(np.mean(attack)) if attack else THRESHOLD   # 숨김 라벨: 같은 종류 학습 공격 라벨 임계값 평균
    return np.array([th.get(l, unseen) for l in lids])

def metrics(df, S, lids, prefix, th=None):
    Y = label_matrix(df, lids)
    P = (S >= (thresholds_for(lids, th) if th else THRESHOLD)).astype(int)
    p, r, f, _ = precision_recall_fscore_support(Y, P, average=None, zero_division=0)
    f05 = precision_recall_fscore_support(Y, (S >= THRESHOLD).astype(int), average=None, zero_division=0)[2]
    has = Y.sum(0) > 0                                     # 이 데이터에 정답이 있는 라벨만 평균
    ap = [average_precision_score(Y[:, j], S[:, j]) for j in range(len(lids)) if has[j]]
    mi = precision_recall_fscore_support(Y, P, average='micro', zero_division=0)
    top = np.array(lids)[S.argmax(1)]
    res = {f'{prefix}_macro_f1': f[has].mean(), f'{prefix}_macro_f1_05': f05[has].mean(), f'{prefix}_micro_f1': mi[2],
           f'{prefix}_macro_p': p[has].mean(), f'{prefix}_macro_r': r[has].mean(),
           f'{prefix}_macro_ap': np.mean(ap) if ap else np.nan,
           f'{prefix}_top1_acc': np.mean([t in ls for t, ls in zip(top, df['labels'])])}
    attack_cols = [j for j, l in enumerate(lids) if l != 'BENIGN']
    if attack_cols:                                        # 공격 판정: 공격 라벨 중 하나라도 임계값을 넘으면 공격
        pred = P[:, attack_cols].any(1)
        true = df['labels'].apply(lambda ls: any(l != 'BENIGN' for l in ls)).values
        res[f'{prefix}_attack_recall'] = pred[true].mean() if true.any() else np.nan
        res[f'{prefix}_fpr'] = pred[~true].mean() if (~true).any() else np.nan
        res[f'{prefix}_separation'] = res[f'{prefix}_attack_recall'] - res[f'{prefix}_fpr']
    per = pd.DataFrame({'label': lids, 'n': Y.sum(0), 'precision': p, 'recall': r, 'f1': f, 'eval': prefix})
    return res, per

def evaluate(path, val_only=False):
    pipe = load_pipe(path)
    res, pers = {}, []
    for kind in KINDS:                                     # 종류마다 따로 추론·채점
        g, P = GROUPS[kind], PREFIX[kind]
        groups = [('val', val_df, g['train'])]
        if not val_only:
            groups += [('seen', g['seen_df'], g['train']),
                       ('held', g['held_df'], g['held'] + ['BENIGN']),
                       ('heldk', g['heldk_df'], g['heldk'] + ['BENIGN']),
                       ('all', test_df, g['train'] + g['held'] + g['heldk'])]
        th = None
        for grp, df, lids in groups:
            if not len(df) or (grp in ('held', 'heldk') and not g[grp]):
                continue
            S = score_matrix(pipe, df.text, lids)
            if grp == 'val' and TUNE_THRESHOLD:
                th = tune_thresholds(df, S, lids)          # 이후 시험 그룹에 그대로 적용
                res.update({f'th_{l}': v for l, v in th.items()})
            r, per = metrics(df, S, lids, f'{P}{grp}', th)
            res.update(r); pers.append(per.assign(kind=kind))
    if len(KINDS) > 1:                                     # 에폭 고르기용: 종류별 검증 지표의 평균
        for m in ['val_macro_ap', 'val_macro_f1']:
            res[m] = float(np.mean([res[f'{PREFIX[k]}{m}'] for k in KINDS]))
    if not val_only:
        ag = pipe(agnews['text'], AG_LABELS, threshold=0.0, batch_size=16)
        res['agnews_acc'] = np.mean([AG_LABELS.index(max(a, key=lambda x: x['score'])['label']) == y
                                     for a, y in zip(ag, agnews['label'])])
    del pipe; torch.cuda.empty_cache()
    return res, (pd.concat(pers) if pers else None)

def show(r):
    return {k: round(float(v), 3) for k, v in r.items()}

for kind, g in GROUPS.items():
    print(f'[{kind}] 평가 데이터: val {len(val_df)} (공격 {int((~val_df["labels"].apply(lambda ls: all(l == "BENIGN" for l in ls))).sum())}) | '
          f'시험 seen {len(g["seen_df"])} / held {len(g["held_df"])} / heldk {len(g["heldk_df"])} / 전체 {len(test_df)}')
    miss = [l for l in g['train'] if not val_df['labels'].apply(lambda ls: l in ls).any()]
    if miss: print(f'  주의: 검증에 없는 학습 라벨 {miss} (임계값은 다른 라벨 평균)')
print('ag_news', len(agnews))

## 6. 기준선 (파인튜닝 전 = k 0)

[하는 일] 원래 모델을 같은 지표로 잰다. 모든 비교의 출발점. 같은 결과 폴더에 이미 있으면 다시 재지 않는다.

[볼 것] `held_*`(제로샷)와 `agnews_acc`가 파인튜닝 뒤에 이 값보다 크게 떨어지면 망각.

In [ ]:
RUNS, CURVES, PER = f'{OUT}/runs.csv', f'{OUT}/curves.csv', f'{OUT}/per_label.csv'
t0 = time.time()
if os.path.exists(RUNS) and os.path.exists(PER):          # 같은 폴더에서 이어 돌릴 때: 저장된 기준선
    base = pd.read_csv(RUNS).query('k == 0').iloc[0].drop(['k', 'seed', 'best_epoch'], errors='ignore').dropna().to_dict()
    base_per = pd.read_csv(PER).query('k == 0')
    print('기준선: runs.csv에서 읽음')
else:
    base, base_per = evaluate(MODEL)
    base_per['k'], base_per['seed'] = 0, 0
    print(f'기준선 {time.time() - t0:.0f}초')
show(base)

## 7. 학습 함수

[하는 일] GLiClass 공식 `train.py`를 그대로 쓰되 두 가지를 더한다.

1. **시드 고정**: `train.py`에는 시드 인자가 없고, 안의 Trainer는 시작할 때 `TrainingArguments.seed`(기본 42)로 시드를 다시 고정한다. 그래서 감싸개(`run_train.py`)가 학습 전 시드를 고정하고, Trainer가 쓰는 `seed`도 우리 seed로 바꾼 뒤 `train.py`를 그대로 실행한다 (dropout, 배치 섞기, 라벨 섞기가 seed마다 달라짐).
2. **에폭마다 체크포인트 + 검증 손실**: `save_steps = eval_steps = 1에폭 스텝 수`. 끝난 뒤 체크포인트마다 우리 지표로 검증해서 `SELECT_BY`가 가장 좋은 에폭을 고른다.

주의: `train.py`의 `type=bool` 인자(`--fp16` 등)는 `False`를 넣어도 참이 된다. 끄려면 아예 넘기지 않는다.

In [ ]:
WRAPPER = os.path.join(WORK, 'run_train.py')
with open(WRAPPER, 'w') as f:
    f.write('''import os, sys, random, runpy
import numpy as np, torch
seed = int(os.environ.get("TRAIN_SEED", "0"))
random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
# Trainer는 시작할 때 args.seed(기본 42)로 시드를 다시 고정하고 배치 순서에도 쓴다: 우리 seed로 바꿈
import gliclass.training as G
_post = G.TrainingArguments.__post_init__
def _post_init(self, *a, **k):
    self.seed = seed
    return _post(self, *a, **k)
G.TrainingArguments.__post_init__ = _post_init
script = sys.argv[1]
sys.argv = sys.argv[1:]
runpy.run_path(script, run_name="__main__")
''')

ARCH = GLiClassModel.from_pretrained(MODEL).config.architecture_type

def train(train_file, seed, out_dir):
    n = len(json.load(open(train_file)))
    steps = max(1, math.ceil(n / BATCH))
    cmd = [sys.executable, WRAPPER, f'{REPO}/train.py', '--model_name', MODEL,
           '--data_path', train_file, '--val_data_path', f'{SPLIT}/val_gliclass.json',
           '--save_path', out_dir, '--problem_type', 'multi_label_classification', '--architecture_type', ARCH,
           '--encoder_lr', str(ENCODER_LR), '--others_lr', str(OTHERS_LR),
           '--num_epochs', str(EPOCHS), '--batch_size', str(BATCH), '--max_length', str(MAX_LEN),
           '--focal_loss_alpha', str(FOCAL_ALPHA), '--focal_loss_gamma', str(FOCAL_GAMMA),
           '--save_steps', str(steps), '--eval_steps', str(steps), '--save_total_limit', str(EPOCHS + 1),
           '--save_only_model', '--max_eval_samples', str(TRAINER_EVAL_N),
           '--logging_steps', '1', '--num_workers', '0', *EXTRA_ARGS]
    proc = subprocess.run(cmd, capture_output=True, text=True, env={**os.environ, 'TRAIN_SEED': str(seed)})
    if proc.returncode != 0:
        print(proc.stdout[-2000:], proc.stderr[-3000:])
        raise RuntimeError(f'학습 실패: {out_dir}')
    ckpts = sorted(glob.glob(f'{out_dir}/checkpoint-*'), key=lambda p: int(p.rsplit('-', 1)[1]))
    log = json.load(open(f'{ckpts[-1]}/trainer_state.json'))['log_history']
    return ckpts, steps, log

def epoch_curve(log, steps):
    """trainer_state.json 기록 → 에폭마다 평균 학습 손실과 검증 손실."""
    tr = pd.DataFrame([x for x in log if 'loss' in x])
    ev = pd.DataFrame([x for x in log if 'eval_loss' in x])
    tr['epoch_i'] = np.ceil(tr.step / steps).astype(int)
    ev['epoch_i'] = (ev.step / steps).round().astype(int)
    return tr.groupby('epoch_i').loss.mean().rename('train_loss').to_frame().join(ev.set_index('epoch_i').eval_loss, how='outer')

## 8. 실험 실행 (k × seed)

[하는 일] k-shot 파일마다 학습 → 에폭별 검증 → 가장 좋은 에폭으로 시험 → 저장. 끝난 실행은 `runs.csv`에 남아 있으면 건너뛰므로 중간에 끊겨도 다시 돌리면 이어서 한다.

[볼 것] 실행마다 한 줄: 고른 에폭, 검증 F1, 시험 seen/held F1, ag_news.

[확인 질문] 고른 에폭(`best_epoch`)이 항상 마지막 에폭이면 무엇을 늘려봐야 할까? 항상 1에폭이면?

In [ ]:
runs = pd.read_csv(RUNS) if os.path.exists(RUNS) else pd.DataFrame([{'k': 0, 'seed': 0, 'best_epoch': 0, **base}])
curves = pd.read_csv(CURVES) if os.path.exists(CURVES) else pd.DataFrame()
per_label = pd.read_csv(PER) if os.path.exists(PER) else base_per
done = set(zip(runs.k, runs.seed))
setting = dict(model=MODEL, epochs=EPOCHS, encoder_lr=ENCODER_LR, others_lr=OTHERS_LR, batch=BATCH, extra=' '.join(EXTRA_ARGS))

for k in K_LIST:
    for seed in SEEDS:
        if (k, seed) in done:
            continue
        if (k, seed) not in train_files:
            print(f'k={k} seed={seed}: 학습 파일 없음'); continue
        t0 = time.time()
        out_dir = os.path.join(WORK, 'runs', MODEL_NAME, f'{VARIANT}_k{k}_s{seed}')
        shutil.rmtree(out_dir, ignore_errors=True)
        ckpts, steps, log = train(train_files[(k, seed)], seed, out_dir)

        cur = epoch_curve(log, steps)
        for ep, ck in enumerate(ckpts, start=1):            # 에폭마다 우리 지표로 검증
            r, _ = evaluate(ck, val_only=True)
            for m, v in r.items():
                cur.loc[ep, m] = v
        best_ep = int(cur[SELECT_BY].idxmax())
        m, per = evaluate(ckpts[best_ep - 1])                # 시험은 고른 에폭으로 한 번만

        if KEEP_MODELS:
            shutil.copytree(ckpts[best_ep - 1], os.path.join(WORK, 'best', MODEL_NAME, f'{VARIANT}{TAG}_k{k}_s{seed}'), dirs_exist_ok=True)
        shutil.rmtree(out_dir, ignore_errors=True)

        runs = pd.concat([runs, pd.DataFrame([{'k': k, 'seed': seed, 'best_epoch': best_ep, **setting, **m}])], ignore_index=True)
        curves = pd.concat([curves, cur.rename_axis('epoch').reset_index().assign(k=k, seed=seed)], ignore_index=True)
        per_label = pd.concat([per_label, per.assign(k=k, seed=seed)], ignore_index=True)
        runs.to_csv(RUNS, index=False); curves.to_csv(CURVES, index=False); per_label.to_csv(PER, index=False)
        line = [f'k={k:2d} seed={seed} ({time.time() - t0:.0f}초) 에폭 {best_ep}/{EPOCHS}', f'val AP {m["val_macro_ap"]:.3f}']
        for kind in KINDS:
            P = PREFIX[kind]
            line.append(f'[{kind}] seen F1 {m.get(P + "seen_macro_f1", np.nan):.3f} held AP {m.get(P + "held_macro_ap", np.nan):.3f} '
                        f'heldk AP {m.get(P + "heldk_macro_ap", np.nan):.3f} '
                        f'판별력 {m.get(P + "seen_separation", np.nan):.3f}')
        print(' | '.join(line + [f'ag_news {m["agnews_acc"]:.3f}']))

## 9. 요약표

[하는 일] k마다 seed 평균 ± 표준편차. k = 0은 기준선.

[볼 것] seed끼리 표준편차가 평균 차이보다 크면 그 차이는 믿기 어렵다.

In [ ]:
BASE_COLS = ['seen_macro_f1', 'seen_macro_f1_05', 'seen_macro_ap', 'seen_separation', 'seen_fpr',
             'held_macro_f1', 'held_macro_ap', 'heldk_macro_ap', 'all_macro_f1']
COLS = ['best_epoch', 'val_macro_ap'] + [PREFIX[k] + c for k in KINDS for c in BASE_COLS] + ['agnews_acc']
COLS = [c for c in COLS if c in runs]
summary = runs.groupby('k')[COLS].agg(['mean', 'std']).round(3)
summary

## 10. 그래프

1. **에폭별 손실 / 검증 AP**: 학습 손실은 내려가는데 검증 손실이 올라가는 지점 = 과적합 시작. 검증 AP는 에폭을 고르는 기준
2. **k에 따른 성능**: 종류(전술/기법)마다 학습 라벨(seen) F1, 늘 숨긴 라벨(held) AP, 판별력. k = 0은 기준선. seen은 오르고 held와 ag_news는 유지되는 게 목표
3. **라벨별 F1**: 어느 라벨이 약한지. 시험에 정답이 있는 라벨만 (seen/held/heldk 그룹)

In [ ]:
import matplotlib.pyplot as plt

ks = sorted(curves.k.unique()) if len(curves) else []
if ks:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for k in ks:
        g = curves[curves.k == k].groupby('epoch')[['train_loss', 'eval_loss', 'val_macro_ap']].mean()
        axes[0].plot(g.index, g.train_loss, marker='o', label=f'k={k} train')
        axes[0].plot(g.index, g.eval_loss, ls='--', marker='s', label=f'k={k} val')
        axes[1].plot(g.index, g.val_macro_ap, marker='o', label=f'k={k}')
    axes[1].axhline(base['val_macro_ap'], color='grey', ls=':', label='before')
    axes[0].set(xlabel='epoch', ylabel='loss (focal)', title='1a. loss per epoch (seed mean)')
    axes[1].set(xlabel='epoch', ylabel='val macro AP', title='1b. validation macro AP per epoch (selection metric)', ylim=(0, 1))
    for ax in axes: ax.legend(fontsize=7, ncol=2); ax.grid(alpha=.3)
    fig.tight_layout(); fig.savefig(f'{OUT}/fig_epochs.png', dpi=200)

m, s = runs.groupby('k')[COLS].mean(), runs.groupby('k')[COLS].std().fillna(0)
fig, ax = plt.subplots(figsize=(8, 4.5))
lines = {'agnews_acc': 'ag_news acc (original skill)'}
for kind in KINDS:
    P = PREFIX[kind]
    lines.update({P + 'seen_macro_f1': f'{kind}: seen macro F1', P + 'held_macro_ap': f'{kind}: held-out macro AP (zero-shot)',
                  P + 'seen_separation': f'{kind}: attack recall - FPR'})
for c_, n in lines.items():
    if c_ in m:
        ax.errorbar(m.index, m[c_], yerr=s[c_], marker='o', capsize=3, label=n)
ax.set(xlabel='examples per label (k), 0 = before fine-tuning', ylabel='score', xticks=list(m.index),
       ylim=(-0.3, 1.05), title=f'2. {MODEL_NAME} few-shot fine-tuning ({VARIANT})')
ax.axhline(0, color='grey', lw=.8); ax.legend(fontsize=7); ax.grid(alpha=.3)
fig.tight_layout(); fig.savefig(f'{OUT}/fig_kshot.png', dpi=200)

pl = per_label[per_label['eval'].str.endswith(('seen', 'held', 'heldk')) & (per_label.n > 0)]   # 정답이 없는 라벨(n=0)은 빼기
h = pl.groupby(['label', 'k']).f1.mean().unstack('k')
h.index = [name.get(l, l) for l in h.index]
fig, ax = plt.subplots(figsize=(1.2 * h.shape[1] + 3, 0.5 * h.shape[0] + 1.5))
im = ax.imshow(h.values, vmin=0, vmax=1, cmap='viridis', aspect='auto')
ax.set(xticks=range(h.shape[1]), xticklabels=h.columns, yticks=range(h.shape[0]), yticklabels=h.index,
       xlabel='k', title='3. test F1 per label (seed mean, labels with test examples)')
for i in range(h.shape[0]):
    for j in range(h.shape[1]):
        if not np.isnan(h.values[i, j]):
            ax.text(j, i, f'{h.values[i, j]:.2f}', ha='center', va='center', color='w' if h.values[i, j] < .6 else 'k', fontsize=8)
fig.colorbar(im); fig.tight_layout(); fig.savefig(f'{OUT}/fig_per_label.png', dpi=200)
print('저장:', sorted(f for f in os.listdir(OUT)))

## 11. 결과 읽는 법

| 관찰 | 뜻 | 다음에 해볼 것 |
|---|---|---|
| seen F1 오름, held AP·ag_news 유지 | 가설 방향과 일치 | 시드·k 늘려 확인 |
| seen F1 오름, held AP 하락 | 학습한 라벨만 외움 | 학습률 낮추기, 에폭 줄이기 |
| ag_news 크게 하락 | 원래 실력 망각 | `EXTRA_ARGS`에 EWC (`--use_ewc --prev_data_path ...`) 또는 일반 데이터 섞기 |
| 학습 손실은 줄고 검증 손실은 오름 | 과적합 | 고른 에폭이 이른지 확인, 학습률 낮추기 |
| 학습 손실도 안 줄어듦 | 학습량 부족 | `ENCODER_LR`·`OTHERS_LR`을 1e-4로, `EPOCHS` 늘리기 (pilot2와 같은 점검) |
| best_epoch이 늘 마지막 | 더 학습할 여지 | `EPOCHS` 늘리기 |
| FPR이 높음 | 정상을 공격으로 많이 잡음 | `THRESHOLD`, BENIGN 예시 수 확인 |

k끼리 제로샷을 비교할 때는 `held_*`를 본다 (`heldk_*`는 k마다 라벨이 다름). 학습률 등을 바꿔 비교할 때는 `TAG`를 바꿔서 결과 폴더를 나눈다 (같은 폴더에서는 끝난 k, seed를 건너뛰기 때문).
`runs.csv`, `curves.csv`, `per_label.csv`, `fig_*.png`를 회의에 가져가면 된다.